# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huseyinTozluyurt/Flyrank-Internship-MachineLearning/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The Rule:** Flag any page that has not been updated in over 180 days (`days_since_last_update > 180`), and score it by multiplying that boolean flag by its 90-day impression volume (`impressions_90d`). This prioritizes the highest-traffic stale pages.

**Reason Codes Output:**
*   *Stale High-Traffic Risk* (Score > 0)
*   *Safe / Recent* (Score = 0)

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)

print("Data loaded. Ready to build baseline rule.")


Data loaded. Ready to build baseline rule.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

We calculate the baseline score, rank the pages in descending order, and export the queue to our outputs directory.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Calculate the heuristic baseline score
df['baseline_score'] = (df['days_since_last_update'] > 180).astype(int) * df['impressions_90d']

# Assign reason codes
df['baseline_reason'] = np.where(df['baseline_score'] > 0, 'Stale High-Traffic Risk', 'Safe / Recent')

# Rank and isolate the queue
baseline_queue = df.sort_values(by='baseline_score', ascending=False)

# Export
os.makedirs("work/outputs", exist_ok=True)
baseline_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)
print("Baseline queue written to work/outputs/baseline_action_score.csv")

Baseline queue written to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Looking closely at the absolute top 20 recommendations from our rigid rule. Out of the top 20, 18 are truly declining (Precision@20 = 0.900).

*   **Action:** Route to the editorial team for immediate content refresh.
*   **Reason Code:** Stale High-Traffic Risk.
*   **Confidence Note:** High confidence for the top ~10 entries due to massive impression volume density (e.g., > 10,000 impressions).
*   **What would make it wrong:** This rule will trigger false positives on highly seasonal pages (e.g., a "Summer Event Guide" that naturally loses traffic in Autumn regardless of content staleness).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


top_20 = baseline_queue.head(20)
p20 = top_20['is_declining_label'].mean()
print(f"Baseline Precision@20: {p20:.3f}")

cols_to_review = ['content_id', 'days_since_last_update', 'impressions_90d', 'avg_position', 'trend_direction']
display(top_20[cols_to_review].head(5))

Baseline Precision@20: 0.900


,content_id,days_since_last_update,impressions_90d,avg_position,trend_direction
16751,content_cf56e2e2e282,194,61678,19.7,down
16514,content_7368877ea310,194,59472,24.8,down
7021,content_1bfaa38ff26c,194,25715,22.2,down
21268,content_0a91db491d14,193,13299,10.5,down
11489,content_5feee3994adb,194,7812,39.0,down


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak Picks:** Reviewing the false positives in the top 20 (e.g., pages with "stable" or "up" trends). The rule fails here because it assumes *all* old content with traffic is decaying. It cannot identify "evergreen" content that maintains a stable, slow-growing long-tail position deep in the search results.

**Leakage Check:** Confirmed that `trend_pct` and `trend_direction` were strictly excluded from the `baseline_score` calculation. The score relies entirely on pre-window signals (`days_since_last_update` and `impressions_90d`), proving the 0.900 Precision@20 is mathematically honest.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Isolate and inspect the false positives (weak picks) in the top 20
weak_picks = top_20[top_20['is_declining_label'] == 0]
print("Weak Picks in Top 20 (False Positives):")
display(weak_picks[cols_to_review])

Weak Picks in Top 20 (False Positives):


,content_id,days_since_last_update,impressions_90d,avg_position,trend_direction
23215,content_bdbec75c1148,194,1316,21.8,stable
8506,content_b65fe2792b44,183,371,16.7,up


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.